In [ ]:
    ############    #############   Connection pooling   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.1 Production Python
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Connection Pooling   #############   ##############   

 =>  Opening a new DB/HTTP connection per request is expensive (TCP handshake, TLS, auth).
       A pool keeps a small set of connections open and hands them out on demand.

 =>  Pool sizing matters: too small -> requests queue and add latency; too large -> you
       exhaust the database's max_connections or the remote service's rate limits.

 =>  When the pool is exhausted, callers should wait with a bounded timeout, not block
       forever -- surface a clear 503/'pool exhausted' error instead of hanging requests.


<img src="images/connection-pool.png" alt="Connection pool diagram: 5 requests, a wait queue, 2 connections, a database">

In [ ]:
import asyncio

class FakeConnection:
    def __init__(self, conn_id: int):
        self.conn_id = conn_id

    async def query(self, sql: str) -> str:
        await asyncio.sleep(0.1)
        return f"conn-{self.conn_id} ran: {sql}"

class SimplePool:
    def __init__(self, size: int):
        self._queue: asyncio.Queue[FakeConnection] = asyncio.Queue()
        for i in range(size):
            self._queue.put_nowait(FakeConnection(i))

    async def acquire(self, timeout: float = 1.0) -> FakeConnection:
        return await asyncio.wait_for(self._queue.get(), timeout=timeout)

    async def release(self, conn: FakeConnection) -> None:
        await self._queue.put(conn)

async def run_query(pool: SimplePool, sql: str):
    conn = await pool.acquire()
    try:
        result = await conn.query(sql)
        print(result)
    finally:
        await pool.release(conn)

async def main():
    pool = SimplePool(size=2)
    await asyncio.gather(*(run_query(pool, f"SELECT {i}") for i in range(5)))

await main()


In [ ]:
 =>  Only 2 FakeConnections exist, so with 5 concurrent queries, requests 3-5 wait for a
       connection to be released before they can run -- exactly the queuing behaviour a
       real pool (SQLAlchemy's QueuePool, asyncpg's Pool) gives you, just made explicit here.

 =>  The try/finally guarantees the connection always goes back to the pool, even if the
       query raises -- a leaked connection is a slow, silent way to take a service down.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Install asyncpg or use SQLAlchemy's async engine against a real (or Dockerized)
           Postgres, and inspect its pool size/overflow settings.

 =>  [ ] Deliberately set pool size to 1 and fire 3 concurrent requests -- observe the
           queuing delay, then fix it by right-sizing the pool.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Not setting a timeout on acquire() -- if every connection is stuck (e.g. a slow query
       upstream), new requests will queue forever with no visible error.

 =>  Sizing the pool without checking the database's own max_connections -- if you run
       multiple app instances, (instances x pool_size) must stay under that ceiling.
